# Multi-source ingestion

Four live open sources, no API keys. Replaces `scripts/ingest.py`.

**This notebook ingests a deliberately small slice** so it executes in under a minute. The graph it reads already holds ~6,850 nodes from seven therapeutic areas; use the pipeline functions directly for a bulk load.

## Setup

Redacted readiness report — `config.py` never prints a secret.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")          # Neo4j emits schema hints we handle
import config
print(config.report())

readiness
  neo4j      uri=neo4j+s://<redacted>.databases.neo4j.io
             user fp=6b611cde  db fp=6b611cde
             password fp=58ebef03
             configured=True
  qdrant     url=https://<redacted>.sa-east-1-0.aws.cloud.qdrant.io
             api_key fp=f3da5a0f
             configured=True
  embedding  provider=bedrock  model=amazon.titan-embed-text-v2:0  dim=1024
             configured=True
  trust      policy=v1.0  floor=0.75  min_tier=ingested


## Fetch from the live APIs

ClinicalTrials.gov v2 · openFDA FAERS · HGNC. The free-text intervention and `medicinalproduct` fields are the point — that is messiness we did not author.

In [2]:
from biomedical.sources import adapters
from biomedical.pipelines import graph_load

print("RxNorm release:", adapters.rxnorm_version())

trials = adapters.fetch_trials("melanoma", limit=10)
aes = adapters.fetch_adverse_events('patient.reaction.reactionmeddrapt:"melanoma"', limit=10)
genes = adapters.fetch_genes(graph_load.GENES)

print(f"\nclinicaltrials.gov  {len(trials)} trials")
print(f"openfda faers       {len(aes)} reports")
print(f"hgnc                {len(genes)} genes")

print("\nsample trial:")
t = trials[0]
print(f"  {t.nct_id}  {t.phase}  {t.status}")
print(f"  conditions:    {list(t.conditions)[:3]}")
print(f"  interventions: {list(t.interventions)[:3]}   <- free text, no identifier")

print("\nsample FAERS drug names (human-entered):")
for a in aes[:4]:
    print("  ", list(a.drug_names)[:3])

RxNorm release: 05-Oct-2026



clinicaltrials.gov  10 trials
openfda faers       10 reports
hgnc                8 genes

sample trial:
  NCT04106167  NA  TERMINATED
  conditions:    ['Advanced Solid Tumor', 'Lymphoma', 'Gastric Cancer']
  interventions: ['Allogeneic natural killer (NK) cell']   <- free text, no identifier

sample FAERS drug names (human-entered):
   ['REVLIMID']
   ['USTEKINUMAB']
   ['CICLOSPORIN', 'PREDNISONE', 'CELLCEPT']
   ['ADALIMUMAB', 'CYCLOSPORINE', 'REMICADE']


## Load into Neo4j

Idempotent on natural keys, so re-running converges.

In [3]:
from biomedical.graph.store import GraphClient
cfg = config.load()

with GraphClient(uri=cfg.neo4j.uri, username=cfg.neo4j.username,
                 password=cfg.neo4j.password.reveal(), database=cfg.neo4j.database) as c:
    graph_load.ensure_schema(c)
    print("trials: ", graph_load.load_trials(c, trials))
    print("reports:", graph_load.load_adverse_events(c, aes))
    print("genes:  ", graph_load.load_genes(c, genes))

  schema: 11 constraints


trials:  10


reports: 10


genes:   8


## Graph composition

Counts are reported **per tier and per source**, never as one total, so a reader can see how much is real.

In [4]:
from biomedical.ui import data

o = data.overview()
print(f"nodes {o['nodes']:,}   relationships {o['rels']:,}\n")
print("by label:")
for r in o["labels"]:
    print(f"  {r['label']:<22} {r['count']:>6,}")
print("\nby source:")
for r in o["sources"]:
    print(f"  {r['source']:<22} {str(r['version'])[:14]:<14} {r['count']:>6,}")
print("\nby provenance tier:")
for r in o["tiers"]:
    print(f"  {r['tier']:<22} {r['count']:>6,}")

nodes 7,819   relationships 10,630

by label:
  Mention                 3,714
  ClinicalTrial           1,208
  Disease                 1,133
  Reaction                  526
  Identifier                451
  AdverseEventReport        409
  Drug                      333
  Conflict                   37
  Gene                        8

by source:
  clinicaltrials.gov     v2              4,233
  openfda.faers          faers           2,146
  mondo                  2026-10-06        908
  rxnorm                 rxnav             472
  conflict-detector      1.0                37
  hgnc                   hgnc-current       23

by provenance tier:
  derived                    37
  ingested                7,782


## The real/synthetic boundary

Stated, not implied. Licence-restricted vocabularies (MedDRA, SNOMED CT, UMLS, DrugBank) are excluded by design so a reader never needs a licence to reproduce this.

In [5]:
print("REAL      drug, trial, adverse-event, gene, disease facts")
print("SYNTHETIC patients, consent status, internal governance policies")
print()
print("relationships by type:")
for r in o["rels_by_type"]:
    print(f"  {r['type']:<26} {r['count']:>6,}")

REAL      drug, trial, adverse-event, gene, disease facts
SYNTHETIC patients, consent status, internal governance policies

relationships by type:
  MENTIONS                    4,915
  STUDIES                     2,267
  REPORTS_REACTION            1,245
  CANDIDATE_FOR                 727
  RESOLVES_TO                   556
  HAS_IDENTIFIER                444
  ASSOCIATED_WITH_REACTION      398
  INVESTIGATES                   71
  REPLACED_BY                     7
